In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import gc
from textwrap import wrap
from random import choices
import seaborn as sns
import pingouin as pg
from scipy.stats import spearmanr
import pickle
import gzip

In [2]:
hippiedata="/home/ajayasha/scratch/Arvind_umd/ECM_proj/hippie/"
genelistdata="/home/ajayasha/scratch/Arvind_umd/ECM_proj/genelist/"
hisigdata="/home/ajayasha/scratch/Arvind_umd/ECM_proj/genelist/hisig/"
UCSCtoilpath="/home/ajayasha/scratch/Arvind_umd/UCSC-TOIL/"
ECMprojpath="/home/ajayasha/scratch/Arvind_umd/ECM_proj/"
RCDprojpath="/home/ajayasha/scratch/Arvind_umd/RCD/"
GDCPANCANpath="/home/ajayasha/scratch/Arvind_umd/GDC-PANCAN/"

In [3]:
gtexmetadata1=pd.read_csv(UCSCtoilpath+"gtex_info/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",sep="\t",index_col=0)
gtexmetadata1["subid"]=gtexmetadata1.index.str.split("-").str[0:2]
gtexmetadata1["subid"]=gtexmetadata1["subid"].str.join("-")
gtexmetadata2=pd.read_csv(UCSCtoilpath+"gtex_info/GTEx_Analysis_v8_Annotations_SubjectPhenotypesDS.txt",sep="\t",index_col=0)
gtexmetadata = gtexmetadata1.merge(gtexmetadata2, left_on='subid', right_on="SUBJID",right_index=True, how='inner', suffixes=('_1', '_2'))
gtexmetadata[["AGE","SMTSISCH","SEX"]]

,AGE,SMTSISCH,SEX
SAMPID,,,
GTEX-1117F-0003-SM-58Q7G,60-69,1188.0,2
GTEX-1117F-0003-SM-5DWSB,60-69,1188.0,2
GTEX-1117F-0003-SM-6WBT7,60-69,1188.0,2
GTEX-1117F-0011-R10a-SM-AHZ7F,60-69,1193.0,2
GTEX-1117F-0011-R10b-SM-CYKQ8,60-69,1193.0,2
...,...,...,...
K-562-SM-E9EZC,50-59,NaN,2
K-562-SM-E9EZI,50-59,NaN,2
K-562-SM-E9EZO,50-59,NaN,2


In [4]:
clindata=pd.read_excel("../TCGA_CDR/TCGA-CDR-SupplementalTableS1.xlsx",index_col=1)

In [5]:
clindata["gender"]=clindata["gender"].str.replace("FEMALE","2",case=True)
clindata["gender"]=clindata["gender"].str.replace("MALE","1",case=True)
clindata["gender"]=clindata["gender"].astype("float")

In [6]:
clindata.index=clindata.index.str.replace("-",".",regex=False)

In [7]:
clindata_gdc=pd.read_csv(GDCPANCANpath+"GDC-PANCAN.basic_phenotype.tsv.gz", sep="\t", index_col=0)
clindata_gdc.index=clindata_gdc.index.str.strip().str[:-1]
clindata_gdc.index=clindata_gdc.index.to_series().apply(lambda x: x.replace("-",".")).values.tolist()
clindata_gdc = clindata_gdc[~clindata_gdc.index.duplicated(keep='first')]
ind_map={}
for i in clindata.index:
    for j in clindata_gdc.index:
        if i==j[:-3]:
            ind_map[i]=j
clindata.index=clindata.index.map(ind_map)

In [8]:
clindata_gdc["Gender"]=clindata_gdc["Gender"].str.replace("Male","2",case=True)
clindata_gdc["Gender"]=clindata_gdc["Gender"].str.replace("Female","1",case=True)
clindata_gdc["Gender"]=clindata_gdc["Gender"].str.replace("not reported","nan",case=True)
clindata_gdc["Gender"]=clindata_gdc["Gender"].str.replace("Unknown","nan",case=True)
clindata_gdc["Gender"]=clindata_gdc["Gender"].astype("float")

In [9]:
gtexmetadata.index=gtexmetadata.index.str.replace("-",".",regex=False)

In [10]:
gtexmetadata[["AGE","SMTSISCH","DTHHRDY","SEX"]]

,AGE,SMTSISCH,DTHHRDY,SEX
SAMPID,,,,
GTEX.1117F.0003.SM.58Q7G,60-69,1188.0,4.0,2
GTEX.1117F.0003.SM.5DWSB,60-69,1188.0,4.0,2
GTEX.1117F.0003.SM.6WBT7,60-69,1188.0,4.0,2
GTEX.1117F.0011.R10a.SM.AHZ7F,60-69,1193.0,4.0,2
GTEX.1117F.0011.R10b.SM.CYKQ8,60-69,1193.0,4.0,2
...,...,...,...,...
K.562.SM.E9EZC,50-59,NaN,NaN,2
K.562.SM.E9EZI,50-59,NaN,NaN,2
K.562.SM.E9EZO,50-59,NaN,NaN,2


In [11]:
clindata_gdc

,program,sample_type_id,sample_type,project_id,Age at Diagnosis in Years,Gender
TCGA.69.7978.01,TCGA,1,Primary Tumor,TCGA-LUAD,59.000000,2.0
TCGA.AR.A24Z.01,TCGA,1,Primary Tumor,TCGA-BRCA,57.000000,1.0
TCGA.D1.A103.01,TCGA,1,Primary Tumor,TCGA-UCEC,87.000000,1.0
TARGET.20.PASRLS.09,TARGET,9,Primary Blood Derived Cancer - Bone Marrow,TARGET-AML,0.816438,1.0
TARGET.20.PASARK.14,TARGET,14,Bone Marrow Normal,TARGET-AML,15.520548,2.0
...,...,...,...,...,...,...
TCGA.3G.AB0T.11,TCGA,11,Solid Tissue Normal,TCGA-THYM,45.000000,2.0
TCGA.44.6775.11,TCGA,11,Solid Tissue Normal,TCGA-LUAD,72.000000,1.0
TCGA.BJ.A0Z0.01,TCGA,1,Primary Tumor,TCGA-THCA,55.000000,2.0
TCGA.A5.A0G2.01,TCGA,1,Primary Tumor,TCGA-UCEC,57.000000,1.0


In [17]:
%%time
#save ssgsea residuals
for file in os.listdir(ECMprojpath+"ssgsea/"):
    if "gsva" in file and file.endswith(".tsv"):
        testfile=pd.read_csv(ECMprojpath+"ssgsea/"+file,index_col=0,sep="\t")
        testexp=testfile.transpose()
        if not testexp.empty:
            if "_Normal Tissue" in file: 
                expfile_wage=pd.concat([testexp,gtexmetadata[["SMTSISCH","AGE","SEX","DTHHRDY"]]],join="inner",axis=1)
                expfile_wage["AGE"]=(expfile_wage["AGE"].str.split("-").str[0].map(float)+expfile_wage["AGE"].str.split("-").str[1].map(float))/2
                expfile_wage["SEX"]=expfile_wage["SEX"].map(float)
                expfile_wage=expfile_wage.dropna()
                depcols=["SMTSISCH","AGE","SEX","DTHHRDY"]
                residuals=pd.DataFrame(index=expfile_wage.index,columns=expfile_wage.columns)
                try:
                    for gene in testexp.columns:
                        res=pg.linear_regression(expfile_wage[depcols],expfile_wage[gene],remove_na=True)
                        residuals[gene]=res.residuals_
                    residuals.drop(depcols,axis=1,inplace=True)
                    residuals.to_csv(ECMprojpath+"ssgsea/residuals/"+file,sep="\t")
                except:
                    print(file)
            else:
                expfile_wage=pd.concat([testexp,clindata_gdc[["Age at Diagnosis in Years","Gender"]]],join="inner",axis=1)
                expfile_wage=expfile_wage.dropna()
                depcols=["Age at Diagnosis in Years","Gender"]
                residuals=pd.DataFrame(index=expfile_wage.index,columns=expfile_wage.columns)
                try:
                    for gene in testexp.columns:
                        res=pg.linear_regression(expfile_wage[depcols],expfile_wage[gene],remove_na=True)
                        residuals[gene]=res.residuals_
                    residuals.drop(depcols,axis=1,inplace=True)
                    residuals.to_csv(ECMprojpath+"ssgsea/residuals/"+file,sep="\t")
                except:
                    print(file)

Acute Lymphoblastic Leukemia_White blood cell_Recurrent Blood Derived Cancer - Bone Marrowandom_ecmcmmed_gsva_mat.tsv
Acute Lymphoblastic Leukemia_White blood cell_Recurrent Blood Derived Cancer - Bone Marrowgsva_supmod_ecmcmmed.tsv
Cells - Transformed Fibroblasts_Skin_Cell Lineandom_ecmcmmed_gsva_mat.tsv
Cells - Ebv-Transformed Lymphocytes_Blood_Cell Linegsva_supmod_ecmcmmed.tsv
Acute Lymphoblastic Leukemia_White blood cell_Recurrent Blood Derived Cancer - Bone Marrowgsva_cshoc.tsv
Cells - Ebv-Transformed Lymphocytes_Blood_Cell Linegsva_cshoc.tsv
Acute Lymphoblastic Leukemia_White blood cell_Primary Blood Derived Cancer - Bone Marrowgsva_supmod_ecmcmmed.tsv
Cells - Leukemia Cell Line (Cml)_Bone Marrow_Cell Lineandom_ecmcmmed_gsva_mat.tsv
Acute Lymphoblastic Leukemia_White blood cell_Primary Blood Derived Cancer - Peripheral Bloodgsva_supmod_ecmcmmed.tsv
Acute Lymphoblastic Leukemia_White blood cell_Primary Blood Derived Cancer - Peripheral Bloodgsva_cshoc.tsv
Cells - Ebv-Transformed L

In [13]:
purity1=pd.read_csv("../subtype_purity/TCGA_mastercalls.abs_tables_JSedit.fixed.txt",sep="\t",index_col=0,header=0)

In [14]:
purity1.index=purity1.index.str.replace("-",".",regex=False)
purity1

,sample,call status,purity,ploidy,Genome doublings,Coverage for 80% power,Cancer DNA fraction,Subclonal genome fraction,solution
array,,,,,,,,,
TCGA.OR.A5J1.01,TCGA-OR-A5J1-01A-11D-A29H-01,called,0.90,2.00,0.0,9.0,0.90,0.02,new
TCGA.OR.A5J2.01,TCGA-OR-A5J2-01A-11D-A29H-01,called,0.89,1.30,0.0,6.0,0.84,0.16,new
TCGA.OR.A5J3.01,TCGA-OR-A5J3-01A-11D-A29H-01,called,0.93,1.27,0.0,5.0,0.89,0.11,new
TCGA.OR.A5J4.01,TCGA-OR-A5J4-01A-11D-A29H-01,called,0.87,2.60,1.0,12.0,0.89,0.08,new
TCGA.OR.A5J5.01,TCGA-OR-A5J5-01A-11D-A29H-01,called,0.93,2.79,1.0,12.0,0.95,0.15,new
...,...,...,...,...,...,...,...,...,...
TCGA.P6.A5OG.01,ACC-TCGA-P6-A5OG-Tumor,maf_call,0.38,1.48,0.0,19.0,0.32,NaN,old
TCGA.R8.A73M.01,LGG-TCGA-R8-A73M-Tumor-SM-4WPJ2,maf_call,1.00,3.87,1.0,16.0,1.00,NaN,old
TCGA.RD.A7BW.01,STAD-TCGA-RD-A7BW-Tumor-SM-4WPA1,maf_call,0.18,1.88,0.0,60.0,0.17,NaN,old


In [15]:
allfiles=[]
for gfile in os.listdir(ECMprojpath+"subtype_purity/BG_deconvolution_without_noncoding_v2/"):
    allfiles.append(pd.read_csv(ECMprojpath+"subtype_purity/BG_deconvolution_without_noncoding_v2/"+gfile,sep="\t",index_col=0))
for gfile in os.listdir(ECMprojpath+"subtype_purity/Kassandra_TCGA/"):
    allfiles.append(pd.read_csv(ECMprojpath+"subtype_purity/Kassandra_TCGA/"+gfile,sep="\t",index_col=0))

In [16]:
deconv=pd.concat(allfiles,join='inner', axis=1)

In [17]:
deconv.columns=deconv.columns.str.replace("-",".",regex=False)

In [18]:
deconv1=deconv.loc["Other"].transpose()

In [15]:
p1=gtexmetadata[["SMTSISCH","AGE","SEX","DTHHRDY"]].copy()
p2=pd.concat([clindata[["age_at_initial_pathologic_diagnosis","gender","type"]],purity1],join="inner",axis=1)
#p1=pd.concat([gtexmetadata[["SMTSISCH","AGE","SEX","DTHHRDY"]],deconv1],join="inner",axis=1)
#p2=pd.concat([clindata[["Age at Diagnosis in Years","Gender"]],deconv1],join="inner",axis=1)

In [16]:
p2

,age_at_initial_pathologic_diagnosis,gender,type,sample,call status,purity,ploidy,Genome doublings,Coverage for 80% power,Cancer DNA fraction,Subclonal genome fraction,solution
TCGA.OR.A5J1.01,58.0,1.0,ACC,TCGA-OR-A5J1-01A-11D-A29H-01,called,0.90,2.00,0.0,9.0,0.90,0.02,new
TCGA.OR.A5J2.01,44.0,2.0,ACC,TCGA-OR-A5J2-01A-11D-A29H-01,called,0.89,1.30,0.0,6.0,0.84,0.16,new
TCGA.OR.A5J3.01,23.0,2.0,ACC,TCGA-OR-A5J3-01A-11D-A29H-01,called,0.93,1.27,0.0,5.0,0.89,0.11,new
TCGA.OR.A5J4.01,23.0,2.0,ACC,TCGA-OR-A5J4-01A-11D-A29H-01,called,0.87,2.60,1.0,12.0,0.89,0.08,new
TCGA.OR.A5J5.01,30.0,1.0,ACC,TCGA-OR-A5J5-01A-11D-A29H-01,called,0.93,2.79,1.0,12.0,0.95,0.15,new
...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA.YZ.A980.01,75.0,1.0,UVM,TCGA-YZ-A980-01A-11D-A39V-01,called,0.46,1.95,0.0,18.0,0.46,0.00,new
TCGA.YZ.A982.01,79.0,2.0,UVM,TCGA-YZ-A982-01A-11D-A39V-01,called,0.97,3.23,1.0,13.0,0.98,0.08,new
TCGA.YZ.A983.01,51.0,2.0,UVM,TCGA-YZ-A983-01A-11D-A39V-01,called,0.94,2.02,0.0,8.0,0.94,0.00,new
TCGA.YZ.A984.01,50.0,2.0,UVM,TCGA-YZ-A984-01A-11D-A39V-01,called,0.93,3.32,1.0,14.0,0.96,0.00,new


In [18]:
%%time
#save ssgsea residuals deconv
for file in os.listdir(ECMprojpath+"ssgsea/"):
    if "gsva" in file and file.endswith(".tsv"):
        testfile=pd.read_csv(ECMprojpath+"ssgsea/"+file,index_col=0,sep="\t")
        testexp=testfile.transpose()
        if not testexp.empty:
            if "_Normal Tissue" in file: 
                continue
            else:
                expfile_wage=pd.concat([testexp,p2[["purity","age_at_initial_pathologic_diagnosis","gender"]]],join="inner",axis=1)
                expfile_wage=expfile_wage.dropna()
                depcols=["purity","age_at_initial_pathologic_diagnosis","gender"]
                residuals=pd.DataFrame(index=expfile_wage.index,columns=expfile_wage.columns)
                try:
                    for gene in testexp.columns:
                        res=pg.linear_regression(expfile_wage[depcols],expfile_wage[gene],remove_na=True)
                        residuals[gene]=res.residuals_
                    residuals.drop(depcols,axis=1,inplace=True)
                    residuals.to_csv(ECMprojpath+"ssgsea/residuals_deconv/"+file,sep="\t")
                except:
                    print(file)

Lung Adenocarcinoma_Lung_Solid Tissue Normalgsva_supmod_ecmcmmed.tsv
Acute Myeloid Leukemia_White blood cell_Primary Blood Derived Cancer - Bone Marrowgsva_supmod_ecmcmmed.tsv
Prostate Adenocarcinoma_Prostate_Solid Tissue Normalandom_ecmcmmed_gsva_mat.tsv
Kidney Papillary Cell Carcinoma_Kidney_Solid Tissue Normalandom_ecmcmmed_gsva_mat.tsv
Head & Neck Squamous Cell Carcinoma_Head and Neck region_Solid Tissue Normalgsva_supmod_ecmcmmed.tsv
Bladder Urothelial Carcinoma_Bladder_Solid Tissue Normalgsva_cshoc.tsv
Esophageal Carcinoma_Esophagus_Solid Tissue Normalandom_ecmcmmed_gsva_mat.tsv
Uterine Corpus Endometrioid Carcinoma_Endometrium_Solid Tissue Normalgsva_cshoc.tsv
Head & Neck Squamous Cell Carcinoma_Head and Neck region_Solid Tissue Normalandom_ecmcmmed_gsva_mat.tsv
Lung Adenocarcinoma_Lung_Solid Tissue Normalgsva_cshoc.tsv
Acute Lymphoblastic Leukemia_White blood cell_Recurrent Blood Derived Cancer - Bone Marrowandom_ecmcmmed_gsva_mat.tsv
Stomach Adenocarcinoma_Stomach_Solid Tissue

In [49]:
p2g

,Other,Age at Diagnosis in Years,Gender
TCGA-FF-8042-01,0.334482,57.0,2.0
TCGA-FF-A7CW-01,0.181861,72.0,1.0
TCGA-GR-A4D6-01,0.153046,54.0,2.0
TCGA-GR-A4D5-01,0.099165,46.0,2.0
TCGA-GS-A9TX-01,0.035121,37.0,1.0
...,...,...,...
TCGA-H6-8124-01,0.592880,56.0,2.0
TCGA-Z5-AAPL-01,0.057409,74.0,2.0
TCGA-RL-AAAS-01,0.378637,60.0,2.0
TCGA-FB-AAQ2-01,0.707626,81.0,2.0


In [51]:
%%time
for file in os.listdir(ECMprojpath+"genexp/fpkm/"):
    if file.endswith(".tsv"):
        print(file)
        testfile=pd.read_csv(ECMprojpath+"genexp/fpkm/"+file,index_col=0,sep="\t")
        testexp=testfile.loc[testfile.median(axis=1)>=1].transpose()
        if "_Normal Tissue" in file: 
            expfile_wage=pd.concat([testexp,p1g],join="inner",axis=1)
            expfile_wage["AGE"]=(expfile_wage["AGE"].str.split("-").str[0].map(float)+expfile_wage["AGE"].str.split("-").str[1].map(float))/2
            expfile_wage["SEX"]=expfile_wage["SEX"].map(float)
            expfile_wage=expfile_wage.dropna()
            depcols=["SMTSISCH","AGE","SEX","DTHHRDY","Other"]
            residuals=pd.DataFrame(index=expfile_wage.index,columns=expfile_wage.columns)
        else:
            expfile_wage=pd.concat([testexp,p2g],join="inner",axis=1)
            expfile_wage=expfile_wage.dropna()
            depcols=["Other","Age at Diagnosis in Years","Gender"]
            residuals=pd.DataFrame(index=expfile_wage.index,columns=expfile_wage.columns)
        try:
            for gene in testexp.columns:
                res=pg.linear_regression(expfile_wage[depcols],expfile_wage[gene],remove_na=True)
                residuals[gene]=res.residuals_
            residuals.drop(depcols,axis=1,inplace=True)
            SCC, pval = spearmanr(residuals,nan_policy="omit")
            ressc=pd.DataFrame(SCC, index=residuals.columns, columns=residuals.columns)
            with gzip.open(file+'_parcorr.pkl.gz', 'wb') as f:
                pickle.dump(ressc, f)
        except:
            print("not "+file)
            continue

Brain - Cerebellum_Brain_Normal Tissue_geneexp.tsv
Clear Cell Sarcoma Of The Kidney_Kidney_Primary Solid Tumor_geneexp.tsv
not Clear Cell Sarcoma Of The Kidney_Kidney_Primary Solid Tumor_geneexp.tsv
Brain - Caudate (Basal Ganglia)_Brain_Normal Tissue_geneexp.tsv
Glioblastoma Multiforme_Brain_Recurrent Tumor_geneexp.tsv
Adrenal Gland_Adrenal Gland_Normal Tissue_geneexp.tsv
Ovarian Serous Cystadenocarcinoma_Ovary_Primary Tumor_geneexp.tsv
Brain Lower Grade Glioma_Brain_Primary Tumor_geneexp.tsv
Ovary_Ovary_Normal Tissue_geneexp.tsv
Lung Adenocarcinoma_Lung_Solid Tissue Normal_geneexp.tsv
Lung Squamous Cell Carcinoma_Lung_Primary Tumor_geneexp.tsv
Artery - Aorta_Blood Vessel_Normal Tissue_geneexp.tsv
Stomach Adenocarcinoma_Stomach_Primary Tumor_geneexp.tsv
Esophageal Carcinoma_Esophagus_Solid Tissue Normal_geneexp.tsv
Colon - Transverse_Colon_Normal Tissue_geneexp.tsv
Brain Lower Grade Glioma_Brain_Recurrent Tumor_geneexp.tsv
Stomach Adenocarcinoma_Stomach_Solid Tissue Normal_geneexp.tsv


CPU times: user 3h 58min, sys: 19min 30s, total: 4h 17min 30s
Wall time: 4h 16min 17s
